# Task 3: Audience Rating Classification

Multi-class classification of maturity rating from genres, format, duration, and release year:
$$y \in \{ \text{TV-MA}, \text{TV-14}, \text{TV-PG}, \text{R}, \text{PG-13}, \text{TV-Y7}, \text{TV-Y}, \text{PG}, \text{TV-G} \}$$

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src.data_loader import get_preprocessed_data

df = get_preprocessed_data('../data/Dataset.csv')
print(f"Loaded {len(df):,} titles with {len(df.columns)} columns.")
print(df['rating'].value_counts().head(10))

Loaded 8,790 titles with 19 columns.
rating
TV-MA    3205
TV-14    2157
TV-PG     861
R         799
PG-13     490
TV-Y7     333
TV-Y      306
PG        287
TV-G      220
NR         79
Name: count, dtype: int64


## 1. Model Training & Comparison

- **Random Forest** with balanced class weights to counter class imbalance.
- **Decision Tree** as an interpretable baseline.

In [2]:
from src.classifier import AudienceRatingClassifier

rf_rating = AudienceRatingClassifier(model_type='rf')
rf_metrics = rf_rating.train(df)
dt_rating = AudienceRatingClassifier(model_type='dt')
dt_metrics = dt_rating.train(df)

pd.DataFrame([
    {"Model": name, "Accuracy": m['accuracy'], "Weighted F1": m['weighted_f1'], "Macro F1": m['macro_f1']}
    for name, m in [("Random Forest", rf_metrics), ("Decision Tree", dt_metrics)]
])

,Model,Accuracy,Weighted F1,Macro F1
0,Random Forest,0.4717,0.4821,0.4546
1,Decision Tree,0.4348,0.4480,0.4323


## 2. Per-Class Breakdown (Random Forest)

In [3]:
report = rf_metrics['classification_report']
pd.DataFrame({c: report[c] for c in rf_metrics['classes']}).T.round(3)

,precision,recall,f1-score,support
PG,0.618,0.596,0.607,57.0
PG-13,0.333,0.520,0.406,98.0
R,0.459,0.625,0.529,160.0
TV-14,0.465,0.451,0.458,432.0
TV-G,0.118,0.227,0.155,44.0
TV-MA,0.690,0.470,0.559,641.0
TV-PG,0.211,0.262,0.234,172.0
TV-Y,0.557,0.803,0.658,61.0
TV-Y7,0.492,0.478,0.485,67.0


## 3. Inference Demo

In [4]:
samples = [
    {"listed_in": "Kids' TV, Anime Series", "type": "TV Show", "duration": "1 Season", "release_year": 2021},
    {"listed_in": "Stand-Up Comedy", "type": "Movie", "duration": "65 min", "release_year": 2020},
    {"listed_in": "Horror Movies, Thrillers", "type": "Movie", "duration": "98 min", "release_year": 2018},
]
for s in samples:
    res = rf_rating.predict(s)
    top3 = ", ".join(f"{k}: {v:.1%}" for k, v in list(res['probabilities'].items())[:3])
    print(f"{s['listed_in']} ({s['type']}) -> {res['prediction']}  [{top3}]")

Kids' TV, Anime Series (TV Show) -> TV-Y7  [TV-Y7: 70.8%, TV-Y: 19.9%, TV-G: 2.9%]
Stand-Up Comedy (Movie) -> TV-MA  [TV-MA: 85.9%, TV-14: 6.9%, TV-PG: 3.9%]
Horror Movies, Thrillers (Movie) -> PG-13  [PG-13: 36.2%, R: 35.6%, TV-MA: 15.4%]


## 4. Findings

- Specialised genres (kids' content, stand-up, horror) produce the most confident predictions.
- Broad genres such as dramas and comedies spread across TV-14 / TV-MA / R; plot synopses or
  transcripts would be needed to separate them further.